### RNN 계열 모델의한계점
- 순차적 : GPU를 활용한 병렬처리 불가
- 장기의존성 : 문장이 길어지면 처음에 읽은 정보가 뒤로 갈수록 희미해진다(기울기 소실과 유사)
###  Attention
- 어떤 단어에 집중할까?
- self-attention : attention을 발전, 입력된 문장안에서 특정 단어가 자기문장내의 다른 단어들과 서로 어떤 관계를 맺고 있는가를 행렬 곱셉한번으로 스스로 연산
    - 이것때문에 순서대로 읽을 필요가 없이 문장의 모든 단어를 한번에 병렬 연산 -- RNN 퇴출되는 계기
### Q,K,V
- Query(Q 질문) : 나는 bank인데 나랑연관된 단어 있어?
- Key(K, 지표) : 나는 money 라는 단어야 내정보표는 이거야
- Value(V, 진짜 값) : 나(money)의 실제 의미는 이거야(최종적으로 섞어들어갈 단어의 실제 의미값)
### 연산 4단계:
1. **점수 계산 ($Q \times K^T$)**: 질문(Q)과 다른 단어들의 정보표(K)를 행렬 내적(Dot-product)하여, 서로 얼마나 연관성이 높은지 어텐션 점수(Score)를 구합니다.
2. **스케일링 (Scaling)**: 차원 크기의 제곱근($\sqrt{d_k}$)으로 점수를 나누어줍니다. 점수가 너무 커져서 Softmax 함수의 미분값이 0이 되는(Gradient Vanishing) 현상을 방지합니다.
3. **가중치 변환 (Softmax)**: 스케일링된 점수를 모두 합치면 1.0(100%)이 되는 확률값인 어텐션 가중치(Attention Weights)로 변환합니다.
4. **문맥 벡터 생성 ($Weights \times V$)**: 이 퍼센트 가중치를 각 단어의 실제 의미(V) 벡터에 곱하고 모두 더하여, 최종적으로 문맥이 완벽히 반영된 하나의 문맥 벡터(Context Vector)를 만듭니다

In [ ]:
# self-attention 연산 구현
# query key vale 행렬 생성
    # 입력된 문장의 단어벡터(정적벡터)가 Linear 연산을 거쳐서 질문, 키, 실제값 텐서변환

import torch
import torch.nn as nn
import math
import torch.nn.functional as F

torch.manual_seed(42)

# 단어 3개짜리 문장 임베딩차원 4차원 q/k/v로 변환도리 차원은 d_k은 2차원
seq_len = 3
embed_size = 4
d_k = 2
# 가상의 입력텐서 x(batch=1,seq=3, embed=4)
x = torch.rand(1,seq_len,embed_size)

# q k v를 만들기위해 선형 레이터 통과
W_Q = nn.Linear(embed_size,d_k,bias=False)
W_K = nn.Linear(embed_size,d_k,bias=False)
W_V = nn.Linear(embed_size,d_k,bias=False)

Q = W_Q(x)
K = W_K(x)
V = W_V(x)

x.shape , Q.shape, K.shape, V.shape


In [ ]:
# Q K V를 이용해서 각 단어가 얼마나 강하게 참고할지 점수를 매기고 최종 문맥 벡터로 만드는과정
# K.transpose(1,2) : (배치, 단어수, 차원)  1번 축(단어수)과 2번축(차원)의 자리름 서로 바꿔줌
# (배치, 차원, 단어수)  -> dot-product 행렬 곱을 할수 있는 전치행렬이 된다
# Q @ K.transpose(1,2)

# 1&2. Q k 전치행렬 내적 후 스케일링
scores = (Q @ K.transpose(1,2)) / math.sqrt(d_k)
print(f'step 1&2 K전치행렬 크기 : {K.transpose(1,2).shape}')
print(f'step 1&2 스케일된 어텐션 점수 크기 : {scores.shape}')

AttributeError: 'Tensor' object has no attribute 'trnaspose'